# Phase 1: Stylometric & Linguistic Feature Extraction
### Classification of Human-Written vs. AI-Generated Text
**National Institute of Technology Sikkim** — Dept. of Computer Science & Engineering  
*Authors:* Pravesh Gedon Sharma (B240019) & Ankit Rawat (B320032)

---
### 🎯 Objectives of this Notebook:
1. **Implement Stylometric Extraction**: As detailed in **Section 4.2** of the report (Lexical diversity, sentence structure, punctuation rates, and readability).
2. **Compute Readability Indices**: Implement and analyze **Flesch Reading Ease (FRE)**, **Flesch-Kincaid Grade Level (FKGL)**, and **Gunning Fog**.
3. **Compare Sentence Burstiness**: Measure $\sigma / \mu$ of sentence lengths to capture stylistic rhythm.
4. **Statistical Significance Testing**: Run Welch's t-tests / Mann-Whitney U tests to verify which stylometric features provide the strongest signal.
5. **Train a Stylometric Baseline Classifier**: Train Logistic Regression and Random Forest strictly on stylometric features to establish an interpretable non-lexical baseline.

## 1. Environment & Imports

In [ ]:
import sys
import os
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

from src.stylometrics import extract_stylometric_features

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
print("Libraries loaded successfully.")

## 2. Load Dataset Subsets for Feature Engineering
We draw a balanced sample of Human and AI texts to extract and visualize stylometric patterns.

In [ ]:
TRAIN_PATH = "../train.csv" if os.path.exists("../train.csv") else "train.csv"
df_raw = pd.read_csv(TRAIN_PATH)

# Stratified sample of 10,000 documents (5,000 Human, 5,000 AI) for fast, robust feature analysis
sample_size = 5000
df_sample = df_raw.groupby("generated", group_keys=False).apply(
    lambda x: x.sample(sample_size, random_state=42)
).reset_index(drop=True)

print(f"Extracted balanced sample: {df_sample['generated'].value_counts().to_dict()}")

## 3. Extract Stylometric Vectors
We compute the full 24-feature stylometric suite for each document:
- **Lexical Diversity**: TTR, Hapax ratio, Avg word length
- **Sentence Structure**: Mean, std, min, max sentence length, burstiness, words per paragraph
- **Punctuation & Syntax**: Rates of commas, semicolons, colons, dashes, quotes, brackets, question/exclamation marks, and capitalization ratio
- **Readability**: Flesch Reading Ease (FRE), Flesch-Kincaid Grade Level (FKGL), Gunning Fog, syllables per word

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas(desc="Extracting stylometrics")

stylo_records = df_sample["text"].progress_apply(extract_stylometric_features).tolist()
df_stylo = pd.DataFrame(stylo_records)
df_stylo["generated"] = df_sample["generated"].values
df_stylo["label"] = df_stylo["generated"].map({0: "Human", 1: "AI"})

print(f"Stylometric feature matrix shape: {df_stylo.shape}")
df_stylo.head()

## 4. Readability Index Comparison (Section 4.2 of Report)
- **Flesch Reading Ease (FRE)**: Higher = easier to read (plain English: 60-70, difficult: 30-50).
- **Flesch-Kincaid Grade Level (FKGL)**: US grade level required to understand.
- **Gunning Fog Index**: Years of formal education needed.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
readability_cols = ["flesch_reading_ease", "flesch_kincaid_grade", "gunning_fog"]
titles = ["Flesch Reading Ease (FRE)", "Flesch-Kincaid Grade (FKGL)", "Gunning Fog Index"]

for ax, col, title in zip(axes, readability_cols, titles):
    sns.boxplot(data=df_stylo, x="label", y=col, ax=ax, palette=["#4C72B0", "#DD8452"], width=0.4)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Score")
plt.tight_layout()
plt.show()

## 5. Burstiness and Sentence Rhythm Analysis
Burstiness is defined as $\sigma / \mu$ of sentence lengths. As stated in Section 4.2: *"Humans mix short and long sentences, so burstiness tends to be higher than for AI text."*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.kdeplot(data=df_stylo, x="burstiness", hue="label", fill=True, common_norm=False, ax=axes[0], palette=["#4C72B0", "#DD8452"])
axes[0].set_title("KDE Distribution of Sentence Burstiness (sigma / mu)", fontweight="bold")
axes[0].set_xlim(0, 2.0)

sns.scatterplot(data=df_stylo.sample(1000, random_state=42), x="mean_sentence_length", y="std_sentence_length", hue="label", alpha=0.6, ax=axes[1], palette=["#4C72B0", "#DD8452"])
axes[1].set_title("Mean vs. Std Dev of Sentence Length", fontweight="bold")
axes[1].set_xlim(0, 50)
axes[1].set_ylim(0, 40)
plt.tight_layout()
plt.show()

## 6. Statistical Significance of Stylometric Features
We compute the difference of means and Cohen's d effect size to rank which stylometric features discriminate best between Human and AI text.

In [ ]:
numeric_features = [c for c in df_stylo.columns if c not in ["generated", "label"]]
stat_rows = []

human_df = df_stylo[df_stylo["generated"] == 0]
ai_df = df_stylo[df_stylo["generated"] == 1]

for feat in numeric_features:
    h_vals = human_df[feat].dropna()
    a_vals = ai_df[feat].dropna()
    
    t_stat, p_val = stats.ttest_ind(h_vals, a_vals, equal_var=False)
    
    # Cohen's d effect size
    pooled_std = np.sqrt((h_vals.std()**2 + a_vals.std()**2) / 2)
    cohens_d = (a_vals.mean() - h_vals.mean()) / pooled_std if pooled_std > 0 else 0
    
    stat_rows.append({
        "feature": feat,
        "human_mean": round(h_vals.mean(), 3),
        "ai_mean": round(a_vals.mean(), 3),
        "diff (AI - Human)": round(a_vals.mean() - h_vals.mean(), 3),
        "cohens_d": round(cohens_d, 3),
        "p_value": f"{p_val:.2e}"
    })

ranking_df = pd.DataFrame(stat_rows).sort_values(by="cohens_d", key=abs, ascending=False)
ranking_df.reset_index(drop=True, inplace=True)
print("Top Stylometric Differentiators by Effect Size:")
display(ranking_df.head(12))

## 7. Training a Stylometric-Only Baseline Classifier
Per Section 6.2 (Ablation Study) of the report:
We evaluate how accurately a pure stylometric model performs *without reading any lexical words*.
All features are scaled using  fitted on the training split only.

In [ ]:
X = df_stylo[numeric_features].fillna(0).values
y = df_stylo["generated"].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Logistic Regression on pure stylometrics
clf_lr = LogisticRegression(max_iter=1000, random_state=42)
clf_lr.fit(X_train_scaled, y_train)
y_pred_lr = clf_lr.predict(X_val_scaled)
y_prob_lr = clf_lr.predict_proba(X_val_scaled)[:, 1]

print("=== Stylometric-Only Logistic Regression Performance ===")
print(classification_report(y_val, y_pred_lr, target_names=["Human", "AI"]))
print(f"ROC-AUC: {roc_auc_score(y_val, y_prob_lr):.4f}")

# Random Forest on pure stylometrics
clf_rf = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
clf_rf.fit(X_train_scaled, y_train)
y_pred_rf = clf_rf.predict(X_val_scaled)
y_prob_rf = clf_rf.predict_proba(X_val_scaled)[:, 1]

print("
=== Stylometric-Only Random Forest Performance ===")
print(classification_report(y_val, y_pred_rf, target_names=["Human", "AI"]))
print(f"ROC-AUC: {roc_auc_score(y_val, y_prob_rf):.4f}")

## 8. Feature Importance Analysis
Which stylometric dimensions carry the most weight in the Random Forest model?

In [ ]:
importances = pd.Series(clf_rf.feature_importances_, index=numeric_features).sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=importances.values[:12], y=importances.index[:12], palette="viridis")
plt.title("Top 12 Most Important Stylometric Features (Random Forest)", fontweight="bold")
plt.xlabel("Relative Importance")
plt.show()

## 9. Key Takeaways & Summary
1. **Stylometry Alone is Discriminative**: Even without reading any specific vocabulary, punctuation habits, sentence burstiness, and readability indices yield significant predictive power.
2. **Interpretability**: Human writing shows higher burstiness (irregular sentence lengths) and distinct punctuation distributions compared to AI-generated text.
3. **Upcoming Step**: In Notebook , we combine these dense stylometric features with word & character n-gram TF-IDF matrices to establish our full Phase 1 baseline.